In [28]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

model_df = pd.read_csv('f1_2023_model_data.csv')


C:\Users\bhavi\AppData\Local\Temp\ipykernel_34892\2255483998.py:7: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  model_df = pd.read_csv('f1_2023_model_data.csv')


In [ ]:
import pandas as pd

track_df = pd.read_csv('track_char.csv')
track_df['EventName'] = track_df['EventName'].str.strip()
track_df['TrackDirection'] = track_df['TrackDirection'].str.strip().str.capitalize()

model_events = set(model_df['EventName'].unique())
track_events = set(track_df['EventName'].unique())

print("In model_df but not track_df:", model_events - track_events)
print("In track_df but not model_df:", track_events - model_events)

In model_df but not track_df: set()
In track_df but not model_df: set()


In [21]:

model_df = model_df.merge(track_df, on='EventName', how='left')

# Re-apply clean-data filter (in case it wasn't already applied to this version of model_df)
model_df = model_df.dropna(subset=['LapTime_Seconds', 'TyreLife', 'TrackTemp', 'AirTemp'])

# Then rebuild the split
sorted_events = model_df[['RoundNumber', 'EventName']].drop_duplicates().sort_values('RoundNumber')
event_order = sorted_events['EventName'].tolist()
print(model_df[track_df.columns].isna().sum())

EventName            0
CircuitLength_km     0
NumCorners           0
NumDRSZones          0
TrackDirection       0
AvgSpeed_kmh         0
ElevationChange_m    0
DownforceLevel       0
dtype: int64


In [22]:
sorted_events = model_df[['RoundNumber', 'EventName']].drop_duplicates().sort_values('RoundNumber')
event_order = sorted_events['EventName'].tolist()

n_test_races = 5
train_events = event_order[:-n_test_races]
test_events = event_order[-n_test_races:]

train_df = model_df[model_df['EventName'].isin(train_events)]
test_df = model_df[model_df['EventName'].isin(test_events)]

In [23]:
features = ['TyreLife', 'Compound', 'FreshTyre', 'Stint', 'TrackTemp', 
            'AirTemp', 'Driver', 'LapNumber',
            'CircuitLength_km', 'NumCorners', 'NumDRSZones', 
            'TrackDirection', 'AvgSpeed_kmh', 'ElevationChange_m', 'DownforceLevel']

X_train = pd.get_dummies(train_df[features], columns=['Compound', 'Driver', 'TrackDirection', 'DownforceLevel'])
X_test = pd.get_dummies(test_df[features], columns=['Compound', 'Driver', 'TrackDirection', 'DownforceLevel'])
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df['LapTime_Seconds']
y_test = test_df['LapTime_Seconds']
print(train_df['LapTime_Seconds'].isna().sum())
print(test_df['LapTime_Seconds'].isna().sum())

0
0


In [24]:

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print(f"MAE: {mae:.3f} seconds")

MAE: 4.392 seconds


In [25]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(15))

CircuitLength_km            0.559175
ElevationChange_m           0.185902
NumCorners                  0.075726
AvgSpeed_kmh                0.050817
AirTemp                     0.024086
Compound_INTERMEDIATE       0.019556
LapNumber                   0.019382
TrackTemp                   0.019170
DownforceLevel_medium       0.015351
DownforceLevel_high         0.008787
Compound_WET                0.005864
NumDRSZones                 0.005845
TyreLife                    0.002864
DownforceLevel_low          0.000813
TrackDirection_Clockwise    0.000793
dtype: float64


In [27]:
import joblib
joblib.dump(model, 'lap_time.joblib')

['lap_time.joblib']

In [29]:
model_df.to_csv('f1_2023_model_ready.csv', index=False)